# Wider feature family: streaming port of the CALIBRATED BATTERY

The highest-value block the streaming model never got. In the batch bank's gain ranking
(`experiments/final_gain_ranking.csv`) **`cal_ar5_var_scan` is #3 overall** and the
`cal_*_scanpeak` family fills the top 40 — none of it is in our 50 streaming features.

It is also **localization done properly**: a shared changepoint `khat` is estimated by
pooling evidence across ALL channels at once. Our earlier localization attempt used a single
variance channel and failed (too noisy to localize with); pooling ~14 channels is a different
proposition.

## How it works
1. ~14 **increment channels**, each centred so `E_ref[increment] ~= 0` under the null.
2. Each divided by `sqrt(Bartlett long-run variance)` measured on the reference. The LRV (not
   the plain variance) is what makes channels comparable: it accounts for autocorrelation, so
   `S` below is a standard random walk under the null whatever the channel's serial dependence.
3. `S = cumsum(normalised increment)` — driftless under the null, gains drift after a break.
4. Per channel: max-segment **scan** `max_k |S[t]-S[k]|/sqrt(t-k)` (a GLR over all changepoints),
   its running peak, the standardized mean, and the segment stat measured from the pooled `khat`.
5. Pooled: `G[k] = sum_ch ((S[t]-S[k])/sqrt(t-k))^2`, `khat = argmax_k G`.

## Streaming vs batch
Batch materialises an `L x L` matrix (O(L^2) — fine offline, impossible per-step). Here the
scan is maximised over a **geometric age grid** (3,4,6,9,... plus the full buffer): O(grid)
per step and exact at the grid points. The scan varies smoothly in log-age, so a log grid
loses little — verified against the batch implementation below.

Tested **stacked on the ramp target** (the confirmed +0.0074 cloud win), since the whole point
is whether feature-side gains ADD to the target-side one.

In [1]:
import os, sys, json, math, time
import numpy as np, pandas as pd

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

REPO = os.path.abspath('../../..')
EPS = 1e-9
LRV_LAGS = 30

# channels: the batch bank's proven winners (ar5_var, ring0, ring12, jump, sign_runs,
# autocorr1, cdf0.75 all rank in the gain top-40) plus the fundamentals.
CHANNELS = ('mean', 'var', 'absdev', 'rank_disp', 'tail', 'autocorr1', 'sign_runs',
            'roughness', 'absdz', 'jump', 'ring0', 'ring12', 'ar5_var', 'cdf0.75')
PER_CH = ('exp', 'scan', 'scanpeak', 'seg')
POOLED = ('cal_glr_strength', 'cal_glr_peak', 'cal_glr_seg_len', 'cal_glr_seg_frac')


def bartlett_lrv(v, lags=LRV_LAGS):
    """Per-series 'natural wobble' scale accounting for autocorrelation (Newey-West)."""
    v = np.asarray(v, np.float64); v = v - v.mean(); n = len(v)
    if n < 10:
        return max(float(v.var()), EPS)
    g0 = float((v * v).mean()); K = min(lags, n // 5); lrv = g0
    for l in range(1, K + 1):
        lrv += 2.0 * (1.0 - l / (K + 1.0)) * float((v[l:] * v[:-l]).mean())
    return float(max(lrv, 0.05 * g0, EPS))


def fit_ar5(z):
    n = len(z)
    if n <= 25:
        return np.zeros(6)
    Xd = np.column_stack([z[4 - i:n - 1 - i] for i in range(5)] + [np.ones(n - 5)])
    beta, *_ = np.linalg.lstsq(Xd, z[5:], rcond=None)
    return beta


def mid_rank(sorted_ref, x):
    n = len(sorted_ref)
    if n == 0:
        return np.zeros(len(np.atleast_1d(x)))
    return 0.5 * (np.searchsorted(sorted_ref, x, 'left')
                  + np.searchsorted(sorted_ref, x, 'right')) / n


def age_grid(maxlen=1024):
    ages, a = [], 3
    while a <= maxlen:
        ages.append(a); a = max(a + 1, int(a * 1.5))
    return np.array(ages, dtype=np.int64)

In [2]:
class CalibBattery:
    """Causal streaming calibrated battery. reset(reference), then update(x) per point.

    NOTE: update() returns a REUSED buffer — callers must .copy() to store it."""

    def __init__(self, max_online=1024):
        self.nch = len(CHANNELS)
        self.ages = age_grid(max_online)
        self.feature_names = [f'cal_{c}_{e}' for c in CHANNELS for e in PER_CH] + list(POOLED)
        self.ncol = len(self.feature_names)
        self._out = np.zeros(self.ncol, np.float64)
        self._max = max_online

    # ---- reference / null calibration ----
    def reset(self, reference):
        r = np.asarray(reference, np.float64)
        self.mu = float(r.mean()) if len(r) else 0.0
        self.sd = float(r.std()) + EPS
        self.sorted_ref = np.sort(r)
        z = (r - self.mu) / self.sd
        self.ar5 = fit_ar5(z)
        self.z_tail = z[-5:] if len(z) >= 5 else np.concatenate([np.zeros(5 - len(z)), z])
        inc_ref = self.channels_batch(r[5:], z[:5], set_null=True) if len(r) > 10 else None
        if inc_ref is None:
            self.lrv = np.ones(self.nch)
        else:
            self.lrv = np.array([bartlett_lrv(inc_ref[c]) for c in CHANNELS])
        self.inv_sd = 1.0 / np.sqrt(np.maximum(self.lrv, EPS))
        self.n = 0
        self.S = np.zeros(self.nch)
        self.hist = np.zeros((self._max + 2, self.nch))     # hist[j] = S after j points
        self.scanpeak = np.zeros(self.nch)
        self.glr_peak = 0.0
        self._zbuf = list(self.z_tail[-5:])
        return self

    def channels_batch(self, x, seed5, set_null=False):
        """Vectorised channels over an array.

        set_null=True estimates the null constants FROM THIS DATA (only correct on the
        reference — reset() does this once). set_null=False reuses the reference-derived
        null, which is what any later call must do: re-estimating on online data would
        silently recalibrate the null to the very shift we are trying to detect."""
        x = np.asarray(x, np.float64); L = len(x)
        z = (x - self.mu) / self.sd
        s = np.asarray(seed5, np.float64)
        if len(s) < 5:
            s = np.concatenate([np.zeros(5 - len(s)), s])
        ze = np.concatenate([s[-5:], z])
        lag1 = ze[4:4 + L]
        lagmat = np.column_stack([ze[4 - i:4 - i + L] for i in range(5)] + [np.ones(L)])
        az = np.abs(z); dz = z - lag1; e5 = z - lagmat @ self.ar5
        if set_null:
            v = float((z * z).mean()) + EPS
            self.ac1 = float((z * lag1).mean() / v)
            self.e_az = float(az.mean())
            self.runs_p = float((np.sign(z) == np.sign(lag1)).mean())
            self.dz_msq = float((dz * dz).mean()) + EPS
            self.dz_mabs = float(np.abs(dz).mean()) + EPS
            self.dz_q95 = float(np.quantile(np.abs(dz), 0.95))
            self.jump_p = float((np.abs(dz) > self.dz_q95).mean())
            self.ring0_p = float((az <= 0.1).mean())
            self.ring12_p = float(((az >= 1) & (az <= 2)).mean())
            self.ar5_msq = float((e5 * e5).mean()) + EPS
        u = mid_rank(self.sorted_ref, x)
        return {
            'mean': z, 'var': z * z - 1.0, 'absdev': az - self.e_az,
            'rank_disp': np.abs(u - 0.5) - 0.25,
            'tail': ((u < 0.05) | (u > 0.95)).astype(float) - 0.10,
            'autocorr1': z * lag1 - self.ac1,
            'sign_runs': (np.sign(z) == np.sign(lag1)).astype(float) - self.runs_p,
            'roughness': dz * dz / self.dz_msq - 1.0,
            'absdz': np.abs(dz) / self.dz_mabs - 1.0,
            'jump': (np.abs(dz) > self.dz_q95).astype(float) - self.jump_p,
            'ring0': (az <= 0.1).astype(float) - self.ring0_p,
            'ring12': ((az >= 1) & (az <= 2)).astype(float) - self.ring12_p,
            'ar5_var': e5 * e5 / self.ar5_msq - 1.0,
            'cdf0.75': (u <= 0.75).astype(float) - 0.75,
        }

    def _increment_one(self, x):
        """The centred channel increments for ONE new point (causal: uses only past z)."""
        z = (x - self.mu) / self.sd
        b = self._zbuf; lag1 = b[-1]
        az = abs(z); dz = z - lag1; adz = abs(dz)
        lagv = np.array([b[-1], b[-2], b[-3], b[-4], b[-5], 1.0])
        e5 = z - float(lagv @ self.ar5)
        u = float(mid_rank(self.sorted_ref, np.array([x]))[0])
        o = np.empty(self.nch)
        o[0] = z
        o[1] = z * z - 1.0
        o[2] = az - self.e_az
        o[3] = abs(u - 0.5) - 0.25
        o[4] = (1.0 if (u < 0.05 or u > 0.95) else 0.0) - 0.10
        o[5] = z * lag1 - self.ac1
        o[6] = (1.0 if (z >= 0) == (lag1 >= 0) else 0.0) - self.runs_p
        o[7] = dz * dz / self.dz_msq - 1.0
        o[8] = adz / self.dz_mabs - 1.0
        o[9] = (1.0 if adz > self.dz_q95 else 0.0) - self.jump_p
        o[10] = (1.0 if az <= 0.1 else 0.0) - self.ring0_p
        o[11] = (1.0 if (1.0 <= az <= 2.0) else 0.0) - self.ring12_p
        o[12] = e5 * e5 / self.ar5_msq - 1.0
        o[13] = (1.0 if u <= 0.75 else 0.0) - 0.75
        b.append(z)
        if len(b) > 5:
            del b[0]
        return o

    def update(self, x):
        self.S += self._increment_one(float(x)) * self.inv_sd
        self.n += 1
        t = self.n
        if t < self.hist.shape[0]:
            self.hist[t] = self.S
        out = self._out

        ages = self.ages[self.ages <= t]
        if len(ages) == 0:
            ages = np.array([t])
        elif ages[-1] != t:
            ages = np.append(ages, t)
        seg = (self.S[None, :] - self.hist[t - ages]) / np.sqrt(ages)[:, None]
        scan = np.abs(seg).max(axis=0)
        np.maximum(self.scanpeak, scan, out=self.scanpeak)

        G = (seg * seg).sum(axis=1)          # pooled evidence per candidate changepoint
        gi = int(G.argmax()); khat_age = float(ages[gi]); segk = seg[gi]

        sqt = math.sqrt(t)
        for c in range(self.nch):
            o = 4 * c
            out[o] = self.S[c] / sqt
            out[o + 1] = scan[c]
            out[o + 2] = self.scanpeak[c]
            out[o + 3] = segk[c]
        nf = float(self.nch)
        strength = (float(G[gi]) - nf) / math.sqrt(2.0 * nf)
        self.glr_peak = max(self.glr_peak, strength)
        b = 4 * self.nch
        out[b] = strength; out[b + 1] = self.glr_peak
        out[b + 2] = math.log1p(khat_age); out[b + 3] = khat_age / t
        return out


proto = CalibBattery()
print(f'{proto.ncol} features: {proto.feature_names[:4]} ... {proto.feature_names[-4:]}')
print('age grid:', proto.ages.tolist())

60 features: ['cal_mean_exp', 'cal_mean_scan', 'cal_mean_scanpeak', 'cal_mean_seg'] ... ['cal_glr_strength', 'cal_glr_peak', 'cal_glr_seg_len', 'cal_glr_seg_frac']
age grid: [3, 4, 6, 9, 13, 19, 28, 42, 63, 94, 141, 211, 316, 474, 711]


In [3]:
# ---- correctness: causality + agreement with the BATCH implementation ----
X = pd.read_parquet(os.path.join(REPO, 'X_train.parquet'))
ids = X.index.get_level_values('id').to_numpy(); per = X['period'].to_numpy()
val = X['value'].to_numpy(np.float64)
uids, starts = np.unique(ids, return_index=True); bounds = np.append(starts, len(ids))

def series(k):
    s, e = bounds[k], bounds[k + 1]; m = per[s:e] == 2
    return val[s:e][~m], val[s:e][m]

# 1. CAUSALITY: truncating the stream must leave earlier rows bit-identical
ref, on = series(0)
full = np.array([CalibBattery().reset(ref).update(v).copy() for v in [on[0]]])  # warm
b1 = CalibBattery().reset(ref); A = np.array([b1.update(v).copy() for v in on[:120]])
b2 = CalibBattery().reset(ref); B = np.array([b2.update(v).copy() for v in on[:60]])
print('causal (prefix-invariant):', np.array_equal(A[:60], B))

# 2. BATCH AGREEMENT: the exact statistic on the full buffer is the age=t grid point,
#    so `exp` (standardized mean) must match the batch formula exactly.
bb = CalibBattery().reset(ref)
rows = np.array([bb.update(v).copy() for v in on])
# set_null=False: reuse the REFERENCE-derived null (re-estimating it on the online data
# would recalibrate the null to the shift we are detecting, and was what made an earlier
# version of this check disagree on the 9 constant-dependent channels).
incs = bb.channels_batch(on, bb.z_tail, set_null=False)
ok = []
for ci, c in enumerate(CHANNELS):
    S = np.cumsum(incs[c] / np.sqrt(bb.lrv[ci]))
    exp_batch = S / np.sqrt(np.arange(1, len(on) + 1))
    ok.append(np.allclose(rows[:, 4 * ci], exp_batch, atol=1e-8))
print(f'exp matches batch formula for {sum(ok)}/{len(CHANNELS)} channels')
print('finite:', np.isfinite(rows).all(), '| shape', rows.shape)

causal (prefix-invariant): True
exp matches batch formula for 14/14 channels
finite: True | shape (416, 60)


In [4]:
# ---- build the calibrated features for every cache row ----
cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
Xbase, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled, base_cols = cache['is_sampled_step'], list(cache['cols'])
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)

CAL_CACHE = 'calib_features.npz'
if os.path.exists(CAL_CACHE):
    cc = np.load(CAL_CACHE); Xcal, csid, ctim = cc['X'], cc['sid'], cc['time']
    print('loaded cached calibrated features', Xcal.shape)
else:
    times = X.index.get_level_values('time').to_numpy()
    Xcal = np.empty((len(sid), proto.ncol), np.float32)
    csid = np.empty(len(sid), np.int64); ctim = np.empty(len(sid), np.int64)
    bat = CalibBattery(); w = 0; t0 = time.time()
    for k in range(len(uids)):
        s, e = bounds[k], bounds[k + 1]; m = per[s:e] == 2
        ref_k = val[s:e][~m]; on_k = val[s:e][m]; to = times[s:e][m]
        if len(on_k) < 1 or (~m).sum() < 8:
            continue                                  # same filter build_cache.py used
        bat.reset(ref_k)
        for i in range(len(on_k)):
            Xcal[w + i] = bat.update(on_k[i])         # float32 assignment copies
        csid[w:w + len(on_k)] = uids[k]; ctim[w:w + len(on_k)] = to
        w += len(on_k)
        if (k + 1) % 2000 == 0:
            print(f'  {k+1:,}/{len(uids):,} | {w:,} rows | {time.time()-t0:.0f}s', flush=True)
    assert w == len(sid), f'row mismatch {w} vs {len(sid)}'
    np.savez(CAL_CACHE, X=Xcal, sid=csid, time=ctim, cols=np.array(proto.feature_names))
    print(f'built {CAL_CACHE} in {time.time()-t0:.0f}s')

assert np.array_equal(csid, sid) and np.array_equal(ctim, tim), 'row misalignment'
Xcal = np.nan_to_num(Xcal, nan=0.0, posinf=0.0, neginf=0.0)
print('aligned OK |', Xcal.shape)

  2,000/10,000 | 1,001,800 rows | 40s


  4,000/10,000 | 2,021,257 rows | 81s


  6,000/10,000 | 3,033,676 rows | 124s


  8,000/10,000 | 4,034,637 rows | 160s


  10,000/10,000 | 5,036,517 rows | 194s


built calib_features.npz in 198s


aligned OK | (5036517, 60)


In [5]:
# ---- CV: does the calibrated battery STACK on the ramp target? ----
from catboost import CatBoostRegressor
idx = pd.read_parquet(os.path.join(REPO, 'y_train_index.parquet'))
tau_row = idx['tau_index'].reindex(np.unique(sid)).reindex(sid).to_numpy()
since = step - tau_row; is_break = tau_row >= 0
W = 40
ramp = np.clip(since / (2.0 * W), 0.0, 1.0); ramp[~is_break] = 0.0
ramp[is_break & (since < 0)] = 0.0
ramp_by_row = pd.Series(ramp, index=index)

REG = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='RMSE',
           random_seed=0, verbose=0, thread_count=-1)

def fp_ramp(train, val):
    m = CatBoostRegressor(**REG)
    m.fit(train.X, ramp_by_row.loc[train.index].to_numpy(), sample_weight=train.w)
    return m.predict(val.X)

Xaug = np.hstack([Xbase, Xcal]).astype(np.float32)
print(f'base {Xbase.shape[1]} + calib {Xcal.shape[1]} = {Xaug.shape[1]} feats')

folds = CV.series_folds(sid, n_splits=5, seed=0)
res = {}
for name, M in [('ramp_base50', Xbase), ('ramp_plus_calib', Xaug)]:
    t = time.time(); print(f'\n=== {name} ({M.shape[1]} feats) ===', flush=True)
    res[name] = CV.run_cv(M, y, sid, index, fp_ramp, folds=folds,
                          train_row_mask=sampled, row_step=step, verbose=False)
    json.dump({k: v.fold_scores.tolist() for k, v in res.items()},
              open('calib_results.json', 'w'), indent=2)
    print(f'{res[name].summary(name)} | {time.time()-t:.0f}s', flush=True)

print('\n=== does the calibrated battery add to the ramp win? ===')
print(CV.paired_compare(res['ramp_plus_calib'], res['ramp_base50'],
                        'ramp_plus_calib', 'ramp_base50'))

base 50 + calib 60 = 110 feats

=== ramp_base50 (50 feats) ===


ramp_base50        mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297 | 175s



=== ramp_plus_calib (110 feats) ===


ramp_plus_calib    mean 0.6076 +/- 0.0111 (sem 0.0050) | OOF 0.6068 | folds: 0.6019  0.6027  0.5985  0.6264  0.6084 | 304s



=== does the calibrated battery add to the ramp win? ===
ramp_plus_calib - ramp_base50: -0.0013 +/- 0.0117 (sem 0.0052, t -0.25, wins 3/5) -> within noise
  per-fold deltas: +0.0079  -0.0017  +0.0027  +0.0057  -0.0212


In [6]:
# which calibrated features does the model actually use?
m = CatBoostRegressor(**REG)
m.fit(Xaug[sampled], ramp[sampled], sample_weight=CV.equal_series_weights(sid[sampled]))
imp = pd.Series(m.get_feature_importance(), index=base_cols + proto.feature_names)
imp = imp.sort_values(ascending=False)
cal_set = set(proto.feature_names)
print('top 20 (calibrated marked *):')
for nm, v in imp.head(20).items():
    print(f'  {"*" if nm in cal_set else " "} {nm:26s} {v:.2f}')
ranks = {nm: int(imp.index.get_loc(nm)) + 1 for nm in proto.feature_names}
best = sorted(ranks.items(), key=lambda kv: kv[1])[:8]
print('\nbest calibrated feature ranks:', best)
print(f'calibrated share of total importance: {imp[list(cal_set)].sum()/imp.sum():.1%}')

top 20 (calibrated marked *):
    n_online                   12.86
    ref_kurt                   4.94
    ref_skew                   4.60
  * cal_ar5_var_scan           4.49
    ref_log_n                  3.42
  * cal_ring12_scanpeak        3.11
    spec_entropy               2.85
  * cal_sign_runs_scanpeak     2.46
  * cal_ar5_var_exp            2.27
  * cal_rank_disp_scanpeak     2.27
    ref_ac1                    2.25
    ar10_resid_logratio        2.22
  * cal_ar5_var_scanpeak       2.14
  * cal_autocorr1_scanpeak     1.95
  * cal_mean_scanpeak          1.86
  * cal_ring0_scanpeak         1.84
  * cal_jump_scanpeak          1.73
  * cal_ar5_var_seg            1.61
  * cal_absdev_scanpeak        1.50
    online_kurt                1.49

best calibrated feature ranks: [('cal_ar5_var_scan', 4), ('cal_ring12_scanpeak', 6), ('cal_sign_runs_scanpeak', 8), ('cal_ar5_var_exp', 9), ('cal_rank_disp_scanpeak', 10), ('cal_ar5_var_scanpeak', 13), ('cal_autocorr1_scanpeak', 14), ('cal_mean_sca